# Real-ESRGAN training pipeline
This is a rough draft containing the main parts for the finetune model. It was not launched and delayed due to technical issues as well as issues related to incompatibility with Kaggle env
The steps outlined in the original [documentation](https://github.com/xinntao/Real-ESRGAN/blob/master/docs/Training.md) were used

## Installation

### Environment setup

In [ ]:
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
%pip install -r requirements.txt
!python setup.py develop

### Downloading pretrained model

In [ ]:
!mkdir -p experiments/pretrained_models
!wget -O experiments/pretrained_models/RealESRGAN_x4plus.pth https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/RealESRGAN_x4plus.pth

## Finetune

### Configuration
inside realesrgan_finetune.yml the entire configuration is contained:

**main-params**
- **name**: experiment name
- **model_type**: RealESRNetModel(stable, faster, less data consumed)/RealESRGANModel(better on faces, slower, needed a lot of data)
- **scale**: upscale coef
- **num_gpu**
- **manual_seed**

**datasets**
- **dataroot_gt / dataroot_lq**: destination paths of paired (real and corrupted) images, ***important** paired image files must have the same name
- **gt_size**: patches size (larger better, but more VRAM)
- **use_flip, use_rot**: additional augmentations
- **dataset_enlarge_ratio**: increases dataset repetition to stabilize learning, can be increased with a small dataset
- **scale**: must be the same as model upscale coef

**network_g**
- **type**: net type (RRDBNet: Residual-in-Residual Dense Blocks)
- **num_feat**: num of features, increases the quality, but slows down and requires more GPU memory
- **num_block**: number of RRDB blocks (more better, but slower, and requires more GPU memory)
- **num_grow_ch**: chanels grow in Dense block
- **scale**: must be the same as model upscale coef

**path**
- **pretrain_network_g**: path to pretrained model
- **strict_load_g**: if true, requires all layers to match when loading
- **resume_state**: gives opportunity to start from checkpoint (kaggle notebook was down)

**train**
- **optim_g**: optimizer type (Adam, etc), learning rate, weight decay (regularization), betas (Adam momentum params)
- **scheduler**:  learning rate scheduler, milestones (Iterations after which lr decreases), gamma (lr scaling coef)
- **total_iter**: total amount of iterations
- **warmup_iter**: gradual increase in lr at the beginning
- **val_freq**: validation frequency (after n iterations validate)
- **pixel_opt**: loss function

### Starting finetune process

In [ ]:
!python -m realesrgan.train -opt /kaggle/working/configs/realesrgan_finetune.yml